# **Setup Environment**

In [ ]:
import sys
sys.path.append('.')

import requests
from app.deploy.main import deploy


API_GATEWAY_URL = deploy()

## **Test**

In [ ]:
def test_root_endpoint() -> None:
    """
    Test the root endpoint (GET /) of the API Gateway.
    """
    resp = requests.get(f"{API_GATEWAY_URL}/")
    print("Status:", resp.status_code)
    print("Response:", resp.json())

In [ ]:
# It's recommended to wait a few seconds to ensure all resources 
# are fully deployed and propagated before testing.
test_root_endpoint()

In [ ]:
def test_generate_upload_url(storage_key: str) -> str:
    """
    Test the /storage/upload-url endpoint by generating a presigned upload URL.

    Parameters
    ----------
    storage_key : str
        Object key (path) in the bucket for which to generate the upload URL.

    Returns
    -------
    str
        Presigned PUT URL for direct object upload to Object Storage.
    """
    
    resp = requests.post(
        f"{API_GATEWAY_URL}/storage/upload-url",
        json={"storage_key": storage_key}
    )
    print("Status:", resp.status_code)
    if resp.ok:
        data = resp.json()
        print("Upload URL:", data["upload_url"])
        print("Storage key:", data["storage_key"])
        return data["upload_url"]
    else:
        print("Error:", resp.json())
        resp.raise_for_status()

In [ ]:
first_upload_url = test_generate_upload_url("test/first-object")

In [ ]:
def upload_file_via_presigned_url(
    upload_url: str,
    content: bytes,
    content_type: str = "text/plain"
) -> None:
    """
    Upload binary content to Object Storage using a presigned URL.

    Parameters
    ----------
    upload_url : str
        Presigned PUT URL generated by the /storage/upload-url endpoint.
    content : bytes, optional
        Binary data to upload.
    content_type : str, optional
        MIME type of the content (default: "text/plain").
    """

    resp = requests.put(
        upload_url,
        data=content,
        headers={"Content-Type": content_type}
    )
    print("Upload result:", resp.status_code)

In [ ]:
upload_file_via_presigned_url(
    upload_url = first_upload_url,
    content = b"Hello from playground!"
)

In [ ]:
def test_get_object_metadata(storage_key: str) -> None:
    """
    Test the /storage/metadata endpoint by retrieving object metadata.

    Parameters
    ----------
    storage_key : str
        Object key (path) in the bucket to inspect.
    """

    resp = requests.get(
        f"{API_GATEWAY_URL}/storage/metadata",
        params={"storage_key": storage_key}
    )
    print("Status:", resp.status_code)
    if resp.ok:
        metadata = resp.json()
        print("Metadata keys:", list(metadata.keys()))
        print("Size:", metadata.get("ContentLength"))
        print("Last modified:", metadata.get("LastModified"))
    else:
        print("Error:", resp.json())
        resp.raise_for_status()

In [ ]:
test_get_object_metadata("test/first-object")

In [ ]:
def test_download_object(storage_key: str) -> None:
    """
    Test the /storage/download endpoint by downloading an object and decoding its base64 content.

    Parameters
    ----------
    storage_key : str
        Object key (path) in the bucket to download.
    """

    resp = requests.get(
        f"{API_GATEWAY_URL}/storage/download",
        params={"storage_key": storage_key}
    )
    print("Status:", resp.status_code)
    if resp.ok:
        print("Downloaded", len(resp.content), "bytes")
        print("Content preview:", resp.content[:50])
    else:
        print("Error:", resp.json())
        resp.raise_for_status()

In [ ]:
test_download_object("test/first-object")

In [ ]:
def test_delete_object(storage_key: str) -> None:
    """
    Test the /storage/delete endpoint by deleting an object from the bucket.

    Parameters
    ----------
    storage_key : str
        Object key (path) in the bucket to delete.
    """
    
    resp = requests.delete(
        f"{API_GATEWAY_URL}/storage/delete",
        params={"storage_key": storage_key}
    )
    print("Status:", resp.status_code)
    if resp.ok:
        print("Object deleted successfully")
        print("Deleted key:", storage_key)
    else:
        print("Error:", resp.json())
        resp.raise_for_status()

In [ ]:
test_delete_object("test/first-object")

In [ ]:
def test_list_objects(max_keys: int = 100) -> None:
    """
    Test the /storage/list endpoint by listing objects in the bucket.

    Parameters
    ----------
    max_keys : int, optional
        Maximum number of objects to list (default: 100).
    """
    
    resp = requests.get(
        f"{API_GATEWAY_URL}/storage/list",
        params={"max_keys": max_keys}
    )
    print("Status:", resp.status_code)
    if resp.ok:
        data = resp.json()
        print("Total objects:", data["count"])
        for obj in data["objects"]:
            print(f"  - {obj['Key']} ({obj['Size']} bytes)")
    else:
        print("Error:", resp.json())
        resp.raise_for_status()

In [ ]:
# Create additional objects to verify that listing returns multiple entries correctly
second_upload_url = test_generate_upload_url("test/second-object")
upload_file_via_presigned_url(
    upload_url = second_upload_url,
    content = b"Greetings from playground!"
)

third_upload_url = test_generate_upload_url("test/third-object")
upload_file_via_presigned_url(
    upload_url = third_upload_url,
    content = b"Hello World!"
)

In [ ]:
test_list_objects()